# DeepRL Monopoly — Builder+DealMaker+Hoarder self-play (Colab Pro run)

Independent run from the primary notebook (`train_colab.ipynb`) — separate
checkpoint path and seed so the two runs don't collide or duplicate each
other. Same branch, same opponent table, same reward shaping.

ASU is not used anywhere in this notebook — no teacher, no labels, no distillation.

## 1. Mount Drive (own checkpoint folder — keeps this run separate from the other one)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepRL_Monopoly_ckpt_pro'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Clone the repo (feature/asu-teacher-distillation branch)

In [ ]:
%cd /content
!rm -rf DeepRL_Monopoly
!git clone --branch feature/asu-teacher-distillation https://github.com/EnzeCbe/monopoly-boom.git DeepRL_Monopoly
%cd DeepRL_Monopoly

## 3. Check GPU + torch (Pro runtime — pick A100/V100 in Runtime > Change runtime type if offered)

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('No GPU — Runtime > Change runtime type > GPU, then re-run this cell.')

## 4. Train

`--seed 7` (different from the primary run's default 42) so this is an
independent trajectory, not a duplicate of the other run — useful for
comparing/ensembling later. Opponent table: `TheBuilder + TheDealMaker +
TheHoarder` (see `monopoly_game_engine/train.py`).

In [ ]:
import os
OUT = f"{CHECKPOINT_DIR}/ddqn_builder_dealmaker_pro.pt"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ddqn --games 5000 --device auto --seed 7 \
  --checkpoint-every 100 \
  --out "{OUT}"

## 5. Resume after a disconnect (same --out path, --resume flag)

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ddqn --games 20000 --device auto --seed 7 --resume \
  --checkpoint-every 100 \
  --out "{OUT}"

## 6. Analyze the per-game log (produced automatically alongside the checkpoint)

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/analyze_run.py "{OUT.rsplit('.', 1)[0]}_games.csv" --window 100

## 7. Quick eval against Builder + DealMaker + Hoarder after training

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/play_game.py \
  --algo ddqn --players 4 --model "{OUT}"